# 05 Lineage - Healthcare

Creates a known Gold-to-Gold derivation so AIDP can capture table and column lineage.


In [ ]:
import re
import oidlUtils

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'healthcare':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pyspark.sql import functions as F

gold_root = f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/{lab_id}"
source_uri = f"{gold_root}/healthcare_patient_utilization/"
target_uri = f"{gold_root}/lineage_demo/"
source = spark.read.format("delta").load(source_uri)
source_count = source.count()
assert source_count > 0, "The Gold source must not be empty"
lineage_demo = (source.groupBy("participant_key")
    .agg(F.count(F.lit(1)).alias("source_row_count"))
    .withColumn("lab_id", F.lit(lab_id))
    .select("participant_key", "lab_id", "source_row_count"))
lineage_demo.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(target_uri)
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_gold.{participant_key}_healthcare_lineage_demo
    (participant_key STRING, lab_id STRING, source_row_count BIGINT)
    USING DELTA LOCATION '{target_uri}'""")
result = spark.read.format("delta").load(target_uri).collect()
assert len(result) == 1 and result[0]["source_row_count"] == source_count
print(f"Lineage verified: aidp_lab.oci_gold.{participant_key}_healthcare_patient_utilization -> aidp_lab.oci_gold.{participant_key}_healthcare_lineage_demo")
